# Прогнозирование почасового трафика на перекрёстке

Итоговый проект по курсу «Анализ временных рядов».

Ноутбук содержит все этапы исследования:

1. Загрузка и очистка данных
2. Разведочный анализ (EDA) и постановка задачи
3. Статистические модели
4. ML-модели с feature engineering
5. Нейросетевые модели
6. Выявление аномалий
7. Финальные выводы

## 1. Загрузка и очистка данных

Исходный файл `data/traffic.csv` содержит почасовые замеры количества
транспортных средств на нескольких перекрёстках. По заданию используем только
перекрёсток `Junction == 1` — для него ряд наиболее длинный и полный.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

DATA_PATH = os.path.join('..', 'data', 'traffic.csv')
raw = pd.read_csv(DATA_PATH)
raw.head()

In [ ]:
# Общая информация о таблице: типы, количество строк, объём памяти
raw.info()

In [ ]:
# Проверка пропусков и дубликатов
print('Пропуски по столбцам:')
print(raw.isna().sum())
print('\nЯвные дубликаты строк:', raw.duplicated().sum())
print('Перекрёстки в данных:', sorted(raw['Junction'].unique().tolist()))
print('Записей по перекрёсткам:')
print(raw['Junction'].value_counts().sort_index())

### Отбор перекрёстка и приведение индекса

Берём `Junction == 1`, приводим `DateTime` к типу `datetime64`, делаем его
индексом и сортируем. Столбец `ID` (порядковый номер) для анализа не нужен,
поэтому удаляем его.

In [ ]:
df = raw.loc[raw['Junction'] == 1].copy()
df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values('DateTime').set_index('DateTime')
df = df.drop(columns=['Junction', 'ID'])
df = df.rename(columns={'Vehicles': 'y'})
df.head()

In [ ]:
# Границы периода наблюдений и шаг ряда
print('Начало ряда :', df.index.min())
print('Конец ряда  :', df.index.max())
print('Наблюдений  :', len(df))

expected_hours = pd.date_range(df.index.min(), df.index.max(), freq='H')
missing_hours = expected_hours.difference(df.index)
print('Ожидалось часов (без пропусков):', len(expected_hours))
print('Пропущенных часов             :', len(missing_hours))

In [ ]:
# Приведение к строго почасовому индексу.
# Если пропусков нет, asfreq просто зафиксирует частоту 'H' в индексе.
# Если появятся NaN — заполним их линейной интерполяцией (разумно для гладкого
# суточного профиля трафика).
df = df.asfreq('H')
print('После asfreq пропусков:', df['y'].isna().sum())
if df['y'].isna().any():
    df['y'] = df['y'].interpolate(method='time')
    print('Пропуски заполнены линейной интерполяцией.')
df.head()

In [ ]:
# Базовая описательная статистика целевой переменной
df['y'].describe().round(2)

In [ ]:
# Среднее по часу суток и дню недели — быстрая прикидка сезонных профилей
by_hour = df.groupby(df.index.hour)['y'].mean().round(1)
by_dow  = df.groupby(df.index.dayofweek)['y'].mean().round(1)
by_dow.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print('Средний трафик по часам суток:')
print(by_hour)
print('\nСредний трафик по дням недели:')
print(by_dow)